# Sub-Region Force Prediction — Validation Notebook

**Goal**: validate the sub-region force prediction pipeline step by step.

| Section | Question |
|---------|----------|
| 0 | Imports & configuration |
| 1 | Data inspection — are force fields consistent with the mass-res pipeline? |
| 2 | Patch sampling — do patches cover the simulation fairly? |
| 3 | CNN correction at HR positions — does the CNN generalise to finer positions? |
| 4 | MLP at HR resolution — can Lagrangian features at HR scale predict ΔF? |
| 5 | Full pipeline — F_LR@HR + ΔF_CNN + ΔF_MLP vs F_HR |
| 6 | Generalisation across snapshots |
| 7 | Patch locality — is the correction spatially local? |

**Key variables** (set in the config cell):
- `F_LR@HR`:   LR force evaluated at ALL HR particle positions  (baseline)
- `F_HR`:      HR PM force at HR positions  (target)
- `ΔF_mass`:   F_HR − F_LR@HR  (total mass-res correction at HR resolution)
- `ΔF_CNN`:    CNN correction at HR patch positions
- `ΔF_MLP`:    MLP correction at HR patch positions
- `F_total`:   F_LR@HR + ΔF_CNN + ΔF_MLP  (full prediction)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import yaml
import pickle
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import jax
import jax.numpy as jnp
import haiku as hk
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.stats import pearsonr

import sys
REPO_ROOT = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT / "pm2nbody"))

from train_lag_massres import compute_subregion_force_pair
from train_subregion_force import (
    load_subregion_snapshot,
    compute_hr_features,
    compute_cnn_at_hr_patch,
    sample_lag_patch,
)
from train_lag_massres import _load_cnn_massres_checkpoint
from jaxpm.lagrangian import get_axis_neighbor_indices, make_lagrangian_corrector

print("imports OK  |  JAX devices:", jax.devices())

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CONFIGURATION — edit this cell
# ═══════════════════════════════════════════════════════════════════
CONFIG_PATH = REPO_ROOT / "configs/subregion_force.yaml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

data_cfg  = SimpleNamespace(**cfg["data"])
model_cfg = SimpleNamespace(**cfg["model"])
train_cfg = SimpleNamespace(**cfg["training"])

DATA_DIR         = Path(data_cfg.data_dir)
MESH_LR          = int(data_cfg.mesh_lr)
MESH_HR          = int(data_cfg.mesh_hr)
BOX_SIZE         = float(data_cfg.box_size)
R                = MESH_HR / MESH_LR
SMOOTH_SIGMA_HR  = float(getattr(data_cfg, "smooth_sigma_hr", 0.5))

SIM_TRAIN  = int(data_cfg.sim_id_train)
SIM_VAL    = int(getattr(data_cfg, "sim_id_val", 1))
SNAP_TRAIN = int(data_cfg.snap_train)
SNAPS_VAL  = list(data_cfg.snaps_val)

USE_STRAIN     = bool(model_cfg.use_strain)
USE_INVARIANTS = bool(model_cfg.use_invariants)
USE_VELOCITY   = bool(model_cfg.use_velocity)
N_SHELL        = int(getattr(model_cfg, "n_shell", 0))
ENV_POOL_MODE  = str(getattr(model_cfg, "env_pool_mode", "mean_var"))
USE_PM_POT     = bool(getattr(model_cfg, "use_pm_potential", True))
PATCH_N        = int(getattr(train_cfg, "patch_n", 32))

CHECKPOINT_DIR = REPO_ROOT / "runs/subregion_force"

print(f"mesh_lr={MESH_LR}  mesh_hr={MESH_HR}  r={R:.0f}  box={BOX_SIZE} Mpc/h")
print(f"σ_hr={SMOOTH_SIGMA_HR} LR-cells  (= {SMOOTH_SIGMA_HR*R:.1f} HR-cells)")
print(f"patch_n={PATCH_N}  ({PATCH_N**3:,} particles per patch)")
print(f"train  sim={SIM_TRAIN}  snap={SNAP_TRAIN}")
print(f"val    sim={SIM_VAL}    snaps={SNAPS_VAL}")

In [ ]:
# ── HR neighbour indices (built once, reused in all sections) ──────────────
neighbor_idx_hr = get_axis_neighbor_indices(MESH_HR)
print(f"HR neighbour index shape: {neighbor_idx_hr.shape}")
print(f"  ({MESH_HR}³ = {MESH_HR**3:,} HR particles, 6 axis neighbours each)")

# ── Force pair helper ─────────────────────────────────────────────────────
# Uses pre-computed delta_f from disk; this helper is for recomputing fresh
# forces (validation / comparison).
_fp_jit = jax.jit(lambda pl, ph: compute_subregion_force_pair(
    pl, ph, MESH_LR, MESH_HR, SMOOTH_SIGMA_HR, 0.0
))

## Section 1 — Data inspection & force pair sanity

**Checks:**
- Position ranges and unit consistency
- `delta_f = f_hr − f_lr_at_hr` magnitude statistics
- Comparison with the mass-res pipeline (stride-subsampled) at matching positions

In [ ]:
snap = load_subregion_snapshot(
    DATA_DIR, SIM_TRAIN, SNAP_TRAIN, MESH_LR, MESH_HR, BOX_SIZE
)
pos_lr = snap["pos_lr"]
pos_hr = snap["pos_hr"]
delta_f_loaded = snap["delta_f"]   # pre-computed by generate_data_subregion.py
a_train = snap["a"]

print(f"a_train = {a_train:.4f}")
print(f"pos_lr:  {pos_lr.shape}  range [{pos_lr.min():.2f}, {pos_lr.max():.2f}] mesh_lr units")
print(f"pos_hr:  {pos_hr.shape}  range [{pos_hr.min():.2f}, {pos_hr.max():.2f}] mesh_lr units")
print(f"delta_f: {delta_f_loaded.shape}  dtype {delta_f_loaded.dtype}")

# Force statistics from disk
df_mag_loaded = np.sqrt(np.sum(np.asarray(delta_f_loaded)**2, axis=-1))
print(f"\nLoaded ΔF statistics:")
print(f"  mean |ΔF| = {df_mag_loaded.mean():.4e}")
print(f"  std  |ΔF| = {df_mag_loaded.std():.4e}")
print(f"  p99  |ΔF| = {np.percentile(df_mag_loaded, 99):.4e}")

# ── Verify: recompute force pair and compare to saved delta_f ─────────────
print("\nRecomputing force pair for comparison …")
f_lr_at_hr, f_hr_recomp, delta_f_recomp = _fp_jit(pos_lr, pos_hr)

f_lr_np    = np.asarray(jax.device_get(f_lr_at_hr))
f_hr_np    = np.asarray(jax.device_get(f_hr_recomp))
df_recomp  = np.asarray(jax.device_get(delta_f_recomp))

# Check agreement between saved and recomputed delta_f
max_diff = np.max(np.abs(df_recomp - np.asarray(delta_f_loaded)))
rms_diff = np.sqrt(np.mean((df_recomp - np.asarray(delta_f_loaded))**2))
print(f"\nSaved vs recomputed delta_f:")
print(f"  max |diff| = {max_diff:.4e}  (should be < 1e-4 for float32)")
print(f"  RMS |diff| = {rms_diff:.4e}")
print("  ✓ PASS" if max_diff < 1e-3 else "  ✗ FAIL — check smooth_sigma_hr in config")

# ── Force magnitude summary ────────────────────────────────────────────────
f_lr_mag = np.sqrt(np.sum(f_lr_np**2, axis=-1))
f_hr_mag = np.sqrt(np.sum(f_hr_np**2, axis=-1))
df_mag   = np.sqrt(np.sum(df_recomp**2, axis=-1))

print(f"\nForce magnitudes (all {MESH_HR**3:,} HR particles):")
print(f"  |F_LR@HR| mean = {f_lr_mag.mean():.4e}")
print(f"  |F_HR|    mean = {f_hr_mag.mean():.4e}")
print(f"  |ΔF|      mean = {df_mag.mean():.4e}  ({df_mag.mean()/f_hr_mag.mean():.1%} of F_HR)")

In [ ]:
# ── Force slab visualization: |F_LR@HR|, |F_HR|, |ΔF| ────────────────────
# Project a thin Eulerian slab of HR particles onto a 2D hex-map.
pos_hr_np = np.asarray(jax.device_get(pos_hr))
slab_z    = MESH_LR // 2
slab_half = max(2, MESH_LR // 20)   # thin slab in mesh_lr units
slab_mask = np.abs(pos_hr_np[:, 2] % MESH_LR - slab_z) < slab_half
print(f"Slab z≈{slab_z}: {slab_mask.sum():,} HR particles")

px = pos_hr_np[slab_mask, 0]
py = pos_hr_np[slab_mask, 1]

vmax = np.percentile(f_hr_mag, 97)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
panels = [
    (f_lr_mag[slab_mask],  "|F_LR@HR|",  "Blues"),
    (f_hr_mag[slab_mask],  "|F_HR|",     "Reds"),
    (df_mag[slab_mask],    "|ΔF_mass|",  "Oranges"),
]
for ax, (vals, label, cmap) in zip(axes, panels):
    hb = ax.hexbin(px, py, C=vals, gridsize=60, cmap=cmap,
                   reduce_C_function=np.mean, vmin=0, vmax=vmax)
    plt.colorbar(hb, ax=ax, label="|F| [mesh_lr]")
    ax.set_title(label); ax.set_xlabel("x [mesh_lr]"); ax.set_ylabel("y")
plt.suptitle(
    f"HR force slab projection  a={a_train:.3f}  "
    f"(slab z={slab_z}±{slab_half}, {MESH_HR}³ HR particles total)",
    fontsize=11
)
plt.tight_layout(); plt.show()

# ── Scatter: |ΔF| vs local overdensity (HR) ──────────────────────────────
from jaxpm.painting import cic_paint
pos_hr_pm = np.mod(pos_hr_np * R, MESH_HR)
delta_hr  = np.asarray(jax.device_get(
    jax.jit(lambda p: __import__('jaxpm.pm', fromlist=['get_delta']).get_delta(
        jnp.array(p), (MESH_HR,)*3
    ))(pos_hr_pm)
))
# Local density at each HR particle via CIC read-out (1+δ)
from jaxpm.painting import cic_read
local_density = np.asarray(jax.device_get(
    jax.jit(lambda d, p: __import__('jaxpm.painting', fromlist=['cic_read']).cic_read(
        jnp.array(d), jnp.array(p)
    ))(delta_hr + 1.0, pos_hr_pm)
))
ss = np.random.default_rng(0).choice(len(df_mag), min(30_000, len(df_mag)), replace=False)
fig, ax = plt.subplots(figsize=(7, 5))
h = ax.hexbin(np.log10(local_density[ss] + 0.1), df_mag[ss],
              gridsize=70, cmap="plasma", norm=LogNorm(), mincnt=1)
plt.colorbar(h, ax=ax, label="particle count")
ax.set_xlabel("log₁₀(1+δ_HR)")
ax.set_ylabel("|ΔF_mass| [mesh_lr]")
ax.set_title("Force correction vs local HR overdensity")
plt.tight_layout(); plt.show()

## Section 2 — Patch sampling

Visualize random Lagrangian patches projected onto the HR density field.
Verify periodic boundary wrapping and check that patch statistics are representative.

In [ ]:
# ── Sample 5 random patches and plot them on the HR density field ──────────
N_PATCHES_SHOW = 5
rng_show = jax.random.PRNGKey(42)
COLORS = ["cyan", "lime", "yellow", "magenta", "orange"]

# Lagrangian positions of ALL HR particles
q_hr = np.stack(np.meshgrid(
    *[np.arange(MESH_HR)] * 3, indexing="ij"
), axis=-1).reshape(-1, 3)   # [mesh_hr³, 3]  Lagrangian grid indices

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: density field with patch outlines (Lagrangian view)
ax = axes[0]
ax.imshow(np.log1p(delta_hr[:, :, MESH_HR//2].T), origin="lower",
          cmap="inferno", extent=[0, MESH_HR, 0, MESH_HR])
ax.set_title("HR density  log(1+δ)  z-slice  [Lagrangian]")

patch_idx_list = []
for i in range(N_PATCHES_SHOW):
    rng_show, key = jax.random.split(rng_show)
    idx = sample_lag_patch(key, MESH_HR, PATCH_N)
    patch_idx_list.append(idx)

    q_patch = q_hr[idx]   # [patch_n³, 3]
    ax.scatter(q_patch[:, 0], q_patch[:, 1],
               c=COLORS[i], s=0.3, alpha=0.4, label=f"patch {i}")

ax.legend(markerscale=8, fontsize=8)
ax.set_xlabel("Lagrangian ix"); ax.set_ylabel("Lagrangian iy")

# Right: |ΔF| map with patch particles highlighted
ax = axes[1]
ax.hexbin(pos_hr_np[:, 0], pos_hr_np[:, 1],
          C=df_mag, gridsize=70, cmap="Greys",
          reduce_C_function=np.mean)
for i, idx in enumerate(patch_idx_list):
    ax.scatter(pos_hr_np[idx, 0], pos_hr_np[idx, 1],
               c=COLORS[i], s=0.5, alpha=0.5)
ax.set_title("|ΔF_mass| map with patch particles [Eulerian]")
ax.set_xlabel("x [mesh_lr]")

plt.suptitle(f"Patch sampling visualization  (patch_n={PATCH_N}, {PATCH_N**3:,} particles)",
             fontsize=11)
plt.tight_layout(); plt.show()

# ── Check: patch ΔF distribution vs global ────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, np.percentile(df_mag, 99), 80)
ax.hist(df_mag, bins=bins, density=True, alpha=0.4, color="gray", label="global")
for i, idx in enumerate(patch_idx_list):
    ax.hist(df_mag[idx], bins=bins, density=True, alpha=0.5,
            color=COLORS[i], label=f"patch {i}")
ax.set_xlabel("|ΔF_mass| [mesh_lr]"); ax.set_ylabel("density")
ax.set_title("Per-patch |ΔF| distribution vs global")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# ── Statistics summary ────────────────────────────────────────────────────
print(f"\n{'Patch':>8} | {'mean |ΔF|':>12} | {'std |ΔF|':>11} | {'SC %':>7}")
print("-" * 50)
feats_hr_tr, det_D_hr_tr = compute_hr_features(
    pos_hr, neighbor_idx_hr, MESH_HR, MESH_LR,
    USE_STRAIN, USE_INVARIANTS,
)
det_D_np = np.asarray(jax.device_get(det_D_hr_tr))
print(f"{'global':>8} | {df_mag.mean():>12.4e} | {df_mag.std():>11.4e} | "
      f"{(det_D_np<0).mean():>6.1%}")
for i, idx in enumerate(patch_idx_list):
    m = df_mag[idx].mean()
    s = df_mag[idx].std()
    sc = (det_D_np[idx] < 0).mean()
    print(f"{'patch '+str(i):>8} | {m:>12.4e} | {s:>11.4e} | {sc:>6.1%}")

## Section 3 — CNN correction at HR positions

The CNN grid is built from LR particles (global context). We evaluate the CNN
scalar potential gradient at HR patch positions — sub-LR-cell precision via
CIC interpolation. If the CNN was trained on LR positions (mass-res pipeline),
we test here whether it generalises to finer HR positions.

In [ ]:
CNN_MODEL  = None
CNN_PARAMS = None
CNN_CKPT   = getattr(model_cfg, "cnn_checkpoint", None)

if CNN_CKPT is not None:
    ckpt_path = str(REPO_ROOT / CNN_CKPT) if not Path(CNN_CKPT).is_absolute() else CNN_CKPT
    try:
        CNN_MODEL, CNN_PARAMS = _load_cnn_massres_checkpoint(ckpt_path)
        n_params = sum(x.size for x in jax.tree_util.tree_leaves(CNN_PARAMS))
        print(f"CNN loaded: {ckpt_path}  ({n_params:,} params)")
    except Exception as e:
        print(f"WARNING: CNN not loaded: {e}")
else:
    print("No cnn_checkpoint in config — CNN correction will be zero in this notebook")

# ── Apply CNN on a validation patch ───────────────────────────────────────
idx_test = patch_idx_list[0]   # use first patch from Section 2
pos_hr_patch = pos_hr[idx_test]

if CNN_MODEL is not None:
    _cnn_jit = jax.jit(lambda pl, ph, a: compute_cnn_at_hr_patch(
        CNN_MODEL, CNN_PARAMS, pl, ph, MESH_LR, a, USE_PM_POT
    ))
    cnn_pred_patch = np.asarray(jax.device_get(
        _cnn_jit(pos_lr, pos_hr_patch, jnp.array(a_train))
    ))
    cnn_mag = np.sqrt(np.sum(cnn_pred_patch**2, axis=-1))
    df_patch = df_recomp[idx_test]
    df_patch_mag = np.sqrt(np.sum(df_patch**2, axis=-1))
    r_cnn = [pearsonr(cnn_pred_patch[:, c], df_patch[:, c])[0] for c in range(3)]

    print(f"\nCNN on HR patch ({len(idx_test):,} particles):")
    print(f"  |ΔF_CNN| mean = {cnn_mag.mean():.4e}")
    print(f"  |ΔF_mass| mean= {df_patch_mag.mean():.4e}")
    print(f"  R(x,y,z) = ({r_cnn[0]:.3f}, {r_cnn[1]:.3f}, {r_cnn[2]:.3f})")

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ci, comp in enumerate(["x", "y", "z"]):
        tgt  = df_patch[:, ci]
        pred = cnn_pred_patch[:, ci]
        lim  = max(abs(np.percentile(tgt, 1)), abs(np.percentile(tgt, 99))) * 1.15
        h    = axes[ci].hexbin(tgt, pred, gridsize=60, cmap="Blues",
                               norm=LogNorm(), mincnt=1,
                               extent=[-lim, lim, -lim, lim])
        plt.colorbar(h, ax=axes[ci])
        axes[ci].plot([-lim, lim], [-lim, lim], "r--", lw=1)
        axes[ci].set_xlabel(f"ΔF_mass_{comp}")
        axes[ci].set_ylabel(f"ΔF_CNN_{comp}")
        axes[ci].set_title(f"{comp}  R={pearsonr(tgt, pred)[0]:.4f}")
    plt.suptitle(
        f"CNN prediction vs ΔF_mass at HR patch positions  a={a_train:.3f}\n"
        "(CNN grid is LR-resolution; evaluated at sub-LR-cell HR positions via CIC)",
        fontsize=10
    )
    plt.tight_layout(); plt.show()
else:
    print("Skipped — no CNN checkpoint loaded.")

## Section 4 — MLP at HR resolution

Load the trained MLP and evaluate on a patch of HR particles.
The MLP uses Lagrangian features computed at HR resolution (mesh_hr grid, mesh_hr units).

In [ ]:
PARAMS_LOADED = None
lag_model = make_lagrangian_corrector(
    hidden_dim=int(model_cfg.hidden_dim),
    n_layers=int(model_cfg.n_layers),
    output_dim=3,
)

# Find latest checkpoint
run_dirs = sorted(CHECKPOINT_DIR.glob("*/"), key=lambda p: p.stat().st_mtime, reverse=True)
for rd in run_dirs:
    for fname in ["best_params.pkl", "final_params.pkl"]:
        pkl = rd / fname
        if pkl.exists():
            with open(pkl, "rb") as fh:
                PARAMS_LOADED = hk.data_structures.to_immutable_dict(pickle.load(fh))
            print(f"Loaded: {pkl}"); break
    if PARAMS_LOADED is not None: break

if PARAMS_LOADED is None:
    print("No checkpoint found in", CHECKPOINT_DIR)

# ── HR Lagrangian features (pre-computed in Section 2) ────────────────────
# feats_hr_tr, det_D_hr_tr were computed in Section 2
feats_hr_np = np.asarray(jax.device_get(feats_hr_tr))
feat_dim    = feats_hr_np.shape[1]
print(f"HR feature dim = {feat_dim}  (HR mesh_hr={MESH_HR}, n_shell={N_SHELL})")
print(f"HR shell-crossing fraction: {(det_D_np < 0).mean():.2%}")

In [ ]:
if PARAMS_LOADED is not None:
    # Select test patch
    idx_test = patch_idx_list[1]

    feats_p = jnp.array(feats_hr_np[idx_test])
    vel_p   = jnp.zeros((len(idx_test), 3))   # use_velocity=False by default

    # MLP predicts ΔF_MLP (= ΔF_mass − ΔF_CNN if two-stage, else ΔF_mass)
    mlp_pred = np.asarray(jax.device_get(
        jax.jit(lag_model.apply)(PARAMS_LOADED, feats_p, vel_p, jnp.array(a_train))
    ))

    # Target: what the MLP was trained on
    df_patch     = df_recomp[idx_test]
    cnn_patch_np = (cnn_pred_patch  # reuse from Section 3 if same patch
                    if CNN_MODEL is not None and idx_test is patch_idx_list[0]
                    else np.zeros_like(df_patch))
    mlp_target   = df_patch - cnn_patch_np   # residual for stage-2, full for stage-1

    err   = mlp_pred - mlp_target
    mse   = float(np.mean(err**2))
    r_mlp = [pearsonr(mlp_pred[:, c], mlp_target[:, c])[0] for c in range(3)]

    print(f"MLP on HR patch ({len(idx_test):,} particles):")
    print(f"  MSE = {mse:.4e}")
    print(f"  R(x,y,z) = ({r_mlp[0]:.3f}, {r_mlp[1]:.3f}, {r_mlp[2]:.3f})")

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    ss_p = np.random.default_rng(0).choice(len(mlp_pred), min(20_000, len(mlp_pred)), replace=False)
    for ci, comp in enumerate(["x", "y", "z"]):
        tgt  = mlp_target[ss_p, ci]
        pred = mlp_pred[ss_p, ci]
        lim  = max(abs(np.percentile(tgt, 1)), abs(np.percentile(tgt, 99))) * 1.15
        h    = axes[ci].hexbin(tgt, pred, gridsize=60, cmap="Greens",
                               norm=LogNorm(), mincnt=1,
                               extent=[-lim, lim, -lim, lim])
        plt.colorbar(h, ax=axes[ci])
        axes[ci].plot([-lim, lim], [-lim, lim], "r--", lw=1)
        axes[ci].set_xlabel(f"Target {comp}")
        axes[ci].set_ylabel(f"ΔF_MLP {comp}")
        axes[ci].set_title(f"{comp}  R={pearsonr(tgt, pred)[0]:.4f}")
    target_label = "ΔF_residual (stage 2)" if CNN_MODEL is not None else "ΔF_mass (stage 1)"
    plt.suptitle(
        f"MLP pred vs {target_label}  |  HR patch  |  a={a_train:.3f}",
        fontsize=11
    )
    plt.tight_layout(); plt.show()
else:
    print("No checkpoint loaded — run train_subregion_force.py first.")

## Section 5 — Full pipeline: F_total vs F_HR

Assemble the complete prediction for all HR particles (full snapshot, not just a patch):

```
F_total = F_LR@HR + ΔF_CNN + ΔF_MLP
```

and compare to F_HR (ground truth). Show force maps, error distributions, and MSE improvement.

In [ ]:
if PARAMS_LOADED is not None:
    import pandas as pd

    # Full-snapshot MLP prediction
    vel_full = jnp.zeros((MESH_HR**3, 3))
    mlp_full_np = np.asarray(jax.device_get(
        jax.jit(lag_model.apply)(
            PARAMS_LOADED, feats_hr_tr, vel_full, jnp.array(a_train)
        )
    ))

    # CNN on full snapshot (if loaded)
    if CNN_MODEL is not None:
        from train_cnn_massres import build_grid_data, compute_cnn_pred
        pos_lr_mod  = jnp.mod(pos_lr, MESH_LR)
        grid_data   = build_grid_data(pos_lr_mod, MESH_LR, USE_PM_POT)
        cnn_full_np = np.asarray(jax.device_get(
            jax.jit(compute_cnn_pred)(
                CNN_MODEL, CNN_PARAMS, grid_data,
                jnp.mod(pos_hr, MESH_LR), jnp.array(a_train)
            )
        ))
    else:
        cnn_full_np = np.zeros_like(mlp_full_np)

    # Total corrected force
    f_total_np  = f_lr_np + cnn_full_np + mlp_full_np
    f_total_mag = np.sqrt(np.sum(f_total_np**2, axis=-1))

    # Errors
    err_lr    = np.sqrt(np.sum((f_lr_np   - f_hr_np)**2, axis=-1))
    err_total = np.sqrt(np.sum((f_total_np - f_hr_np)**2, axis=-1))

    mse_lr    = float(np.mean((f_lr_np   - f_hr_np)**2))
    mse_total = float(np.mean((f_total_np - f_hr_np)**2))
    improv    = 1.0 - mse_total / mse_lr
    frac_imp  = float(np.mean(err_total < err_lr))

    stage_label = ("F_LR@HR + ΔF_CNN + ΔF_MLP"
                   if CNN_MODEL is not None else "F_LR@HR + ΔF_MLP")

    print(f"Full pipeline  [{stage_label}]")
    print(f"  MSE(F_LR@HR,  F_HR) = {mse_lr:.4e}   (baseline)")
    print(f"  MSE(F_total, F_HR)  = {mse_total:.4e}")
    print(f"  MSE improvement     = {improv:.1%}")
    print(f"  Particles improved  = {frac_imp:.1%}")

    # ── Force magnitude distributions ─────────────────────────────────────
    bins = np.linspace(0, np.percentile(f_hr_mag, 99.5), 120)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(f_lr_mag,    bins=bins, alpha=0.5, density=True, color="steelblue", label="|F_LR@HR|")
    ax.hist(f_total_mag, bins=bins, alpha=0.5, density=True, color="seagreen",  label=f"|{stage_label}|")
    ax.hist(f_hr_mag,    bins=bins, alpha=0.5, density=True, color="tomato",    label="|F_HR|")
    for c, m in [(("steelblue"), f_lr_mag.mean()),
                 (("seagreen"),  f_total_mag.mean()),
                 (("tomato"),    f_hr_mag.mean())]:
        ax.axvline(m, color=c, ls="--", lw=1.2, alpha=0.9)
    ax.set_xlabel("|F| [mesh_lr]"); ax.set_ylabel("density")
    ax.set_title(f"Force magnitude distributions  a={a_train:.3f}  ({MESH_HR**3:,} HR particles)")
    ax.legend(); plt.tight_layout(); plt.show()

    # ── Force maps (slab) ─────────────────────────────────────────────────
    vmax_f = np.percentile(f_hr_mag, 97)
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, (mag, label, cmap) in zip(axes, [
        (f_lr_mag[slab_mask],    "F_LR@HR",    "Blues"),
        (f_total_mag[slab_mask], stage_label,  "Greens"),
        (f_hr_mag[slab_mask],    "F_HR (ref)", "Reds"),
    ]):
        hb = ax.hexbin(px, py, C=mag, gridsize=60, cmap=cmap,
                       reduce_C_function=np.mean, vmin=0, vmax=vmax_f)
        plt.colorbar(hb, ax=ax, label="|F|")
        ax.set_title(label); ax.set_xlabel("x")
    plt.suptitle(
        f"{stage_label}\n"
        f"MSE improvement: {improv:.1%}  |  {frac_imp:.1%} of particles improved",
        fontsize=10
    )
    plt.tight_layout(); plt.show()

    # ── Per-particle improvement scatter ──────────────────────────────────
    lim_err = np.percentile(err_lr, 99)
    fig, ax = plt.subplots(figsize=(7, 6))
    h = ax.hexbin(err_lr, err_total, gridsize=80, cmap="plasma",
                  norm=LogNorm(), mincnt=1, extent=[0, lim_err, 0, lim_err])
    plt.colorbar(h, ax=ax, label="particle count")
    ax.plot([0, lim_err], [0, lim_err], "w--", lw=1.5, label="no change")
    ax.set_xlabel("|F_LR@HR − F_HR|  (before)")
    ax.set_ylabel(f"|F_total − F_HR|  (after)")
    ax.set_title(f"Per-particle error  ({frac_imp:.1%} improved)")
    ax.legend(); plt.tight_layout(); plt.show()
else:
    print("No checkpoint — skipping Section 5.")

## Section 6 — Generalisation across snapshots

Loop over `SNAPS_VAL` (different scale factors `a`) and compute MSE improvement for each.
This tests temporal generalization — the MLP was trained on a single snapshot.

In [ ]:
if PARAMS_LOADED is not None:
    import pandas as pd
    rows = []
    for snap_id in SNAPS_VAL:
        snap_v = load_subregion_snapshot(
            DATA_DIR, SIM_VAL, snap_id, MESH_LR, MESH_HR, BOX_SIZE
        )
        pv_lr = snap_v["pos_lr"]
        pv_hr = snap_v["pos_hr"]
        av    = snap_v["a"]

        # Force pair
        fv_lr, fv_hr, dfv = _fp_jit(pv_lr, pv_hr)
        fv_lr_np = np.asarray(jax.device_get(fv_lr))
        fv_hr_np = np.asarray(jax.device_get(fv_hr))

        # HR features
        fv_feats, fv_det_D = compute_hr_features(
            pv_hr, neighbor_idx_hr, MESH_HR, MESH_LR, USE_STRAIN, USE_INVARIANTS
        )

        # CNN
        if CNN_MODEL is not None:
            from train_cnn_massres import build_grid_data, compute_cnn_pred
            gd_v = build_grid_data(jnp.mod(pv_lr, MESH_LR), MESH_LR, USE_PM_POT)
            cnn_v = np.asarray(jax.device_get(
                jax.jit(compute_cnn_pred)(CNN_MODEL, CNN_PARAMS, gd_v,
                                          jnp.mod(pv_hr, MESH_LR), jnp.array(av))
            ))
        else:
            cnn_v = np.zeros((MESH_HR**3, 3), dtype=np.float32)

        # MLP
        vel_v = jnp.zeros((MESH_HR**3, 3))
        mlp_v = np.asarray(jax.device_get(
            jax.jit(lag_model.apply)(PARAMS_LOADED, fv_feats, vel_v, jnp.array(av))
        ))

        f_tot_v = fv_lr_np + cnn_v + mlp_v
        err_v_lr  = np.sqrt(np.sum((fv_lr_np - fv_hr_np)**2, axis=-1))
        err_v_tot = np.sqrt(np.sum((f_tot_v  - fv_hr_np)**2, axis=-1))
        mse_lr_v  = float(np.mean((fv_lr_np - fv_hr_np)**2))
        mse_tot_v = float(np.mean((f_tot_v  - fv_hr_np)**2))

        rows.append({
            "snap": snap_id, "a": av,
            "SC %": float(np.mean(np.asarray(jax.device_get(fv_det_D)) < 0)) * 100,
            "MSE_LR": mse_lr_v, "MSE_total": mse_tot_v,
            "MSE_improv %": (1 - mse_tot_v / mse_lr_v) * 100,
            "frac_improved": float(np.mean(err_v_tot < err_v_lr)) * 100,
        })
        del snap_v, fv_lr, fv_hr, dfv, fv_feats, fv_det_D, mlp_v

    df_gen = pd.DataFrame(rows)
    pd.set_option("display.float_format", lambda x: f"{x:.4f}")
    print(df_gen.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(df_gen["a"], df_gen["MSE_improv %"], "o-", color="seagreen", lw=2)
    axes[0].axhline(0, color="k", ls="--", lw=0.8)
    axes[0].set_xlabel("a"); axes[0].set_ylabel("MSE improvement %")
    axes[0].set_title("MSE improvement vs scale factor"); axes[0].grid(alpha=0.3)

    axes[1].plot(df_gen["a"], df_gen["frac_improved"], "o-", color="tomato", lw=2)
    axes[1].axhline(50, color="k", ls="--", lw=0.8, label="50% baseline")
    axes[1].set_xlabel("a"); axes[1].set_ylabel("% particles improved")
    axes[1].set_title("Fraction of particles improved"); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.suptitle(f"Generalisation across snapshots  [{stage_label}]", fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print("No checkpoint — skipping Section 6.")

## Section 7 — Patch locality: is the correction spatially consistent?

Key question: does the model predict similar-quality corrections across different patch
locations (dense halos vs. voids)? If patch-dependent performance is low, the model
is not robust to the "which sub-region you pick" choice.

We evaluate 20+ random patches and measure improvement per patch, then stratify by
patch properties (SC fraction, mean density, mean |ΔF|).

In [ ]:
if PARAMS_LOADED is not None:
    N_LOCALITY_PATCHES = 30
    rng_loc = jax.random.PRNGKey(999)
    patch_stats = []

    # Pre-compute CNN for full snapshot (reuse if already computed above)
    cnn_available = CNN_MODEL is not None

    for pi in range(N_LOCALITY_PATCHES):
        rng_loc, key = jax.random.split(rng_loc)
        idx = sample_lag_patch(key, MESH_HR, PATCH_N)

        feats_pi  = jnp.array(feats_hr_np[idx])
        vel_pi    = jnp.zeros((len(idx), 3))
        mlp_pi    = np.asarray(jax.device_get(
            lag_model.apply(PARAMS_LOADED, feats_pi, vel_pi, jnp.array(a_train))
        ))
        cnn_pi    = cnn_full_np[idx] if cnn_available else np.zeros_like(mlp_pi)

        f_lr_pi   = f_lr_np[idx]
        f_hr_pi   = f_hr_np[idx]
        f_tot_pi  = f_lr_pi + cnn_pi + mlp_pi

        err_lr_pi  = np.sqrt(np.sum((f_lr_pi  - f_hr_pi)**2, axis=-1))
        err_tot_pi = np.sqrt(np.sum((f_tot_pi - f_hr_pi)**2, axis=-1))
        mse_lr_pi  = float(np.mean((f_lr_pi  - f_hr_pi)**2))
        mse_tot_pi = float(np.mean((f_tot_pi - f_hr_pi)**2))

        df_pi_mag  = df_mag[idx]
        sc_pi      = float((det_D_np[idx] < 0).mean())

        patch_stats.append({
            "mse_improv %":    (1 - mse_tot_pi / mse_lr_pi) * 100,
            "frac_improved":   float(np.mean(err_tot_pi < err_lr_pi)),
            "mean_df":         df_pi_mag.mean(),
            "sc_frac":         sc_pi,
            "mean_dens":       local_density[idx].mean() if 'local_density' in dir() else 0.0,
        })

    df_loc = pd.DataFrame(patch_stats)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Improvement vs |ΔF| of the patch
    axes[0].scatter(df_loc["mean_df"], df_loc["mse_improv %"],
                    c=df_loc["sc_frac"], cmap="coolwarm", s=60, edgecolors="k", lw=0.4)
    axes[0].axhline(0, color="k", ls="--", lw=0.8)
    axes[0].set_xlabel("mean |ΔF_mass| in patch")
    axes[0].set_ylabel("MSE improvement %")
    axes[0].set_title("Improvement vs |ΔF| (colour = SC fraction)")
    axes[0].grid(alpha=0.3)

    # Improvement vs shell-crossing fraction
    axes[1].scatter(df_loc["sc_frac"] * 100, df_loc["mse_improv %"],
                    c=df_loc["mean_df"], cmap="viridis", s=60, edgecolors="k", lw=0.4)
    axes[1].axhline(0, color="k", ls="--", lw=0.8)
    axes[1].set_xlabel("SC % in patch")
    axes[1].set_ylabel("MSE improvement %")
    axes[1].set_title("Improvement vs shell-crossing fraction")
    axes[1].grid(alpha=0.3)

    # Distribution of per-patch improvements
    axes[2].hist(df_loc["mse_improv %"], bins=15, color="seagreen", alpha=0.8)
    axes[2].axvline(df_loc["mse_improv %"].mean(), color="tomato", lw=2,
                    label=f"mean={df_loc['mse_improv %'].mean():.1f}%")
    axes[2].axvline(0, color="k", ls="--", lw=0.8)
    axes[2].set_xlabel("MSE improvement % per patch")
    axes[2].set_title(f"Distribution across {N_LOCALITY_PATCHES} random patches")
    axes[2].legend(); axes[2].grid(alpha=0.3)

    plt.suptitle("Patch locality analysis: is improvement spatially uniform?", fontsize=11)
    plt.tight_layout(); plt.show()

    print(f"\nPatch improvement statistics ({N_LOCALITY_PATCHES} patches):")
    print(f"  Mean MSE improvement : {df_loc['mse_improv %'].mean():.1f}%")
    print(f"  Std  MSE improvement : {df_loc['mse_improv %'].std():.1f}%")
    print(f"  Min  MSE improvement : {df_loc['mse_improv %'].min():.1f}%")
    print(f"  Patches with improv>0: {(df_loc['mse_improv %'] > 0).mean():.1%}")
    print()
    r_sc   = pearsonr(df_loc["sc_frac"],  df_loc["mse_improv %"])[0]
    r_df   = pearsonr(df_loc["mean_df"],  df_loc["mse_improv %"])[0]
    print(f"  R(SC_frac, improvement) = {r_sc:.3f}")
    print(f"  R(mean|ΔF|, improvement)= {r_df:.3f}")
    print()
    print("Interpretation:")
    print("  Low R → improvement is uniform across patch types (robust model)")
    print("  High R(SC_frac) → model helps more in shell-crossing regions")
else:
    print("No checkpoint — skipping Section 7.")